# Part 2: Exploratory Data Analysis & Prescriptive Insights
**Team:** GenCore | **Lead:** Trịnh Hoàng Tú

**Objective:** Extract actionable business insights from 11 datasets to drive growth and operational efficiency.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import sys
import warnings
warnings.filterwarnings('ignore')

# Style
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 6)

# === Environment Detection ===
IS_KAGGLE = os.path.exists('/kaggle/input')

if IS_KAGGLE:
    DATA_DIR = None
    for root, dirs, files in os.walk('/kaggle/input'):
        if 'sales.csv' in files:
            DATA_DIR = root + '/'
            break
    if DATA_DIR is None:
        raise FileNotFoundError('Could not find sales.csv under /kaggle/input')
else:
    DATA_DIR = '../data/raw/'
    sys.path.append(os.path.abspath('..'))

print(f"Environment: {'Kaggle' if IS_KAGGLE else 'Local'}")
print(f"Data dir: {DATA_DIR}")

## 0 — Load All Data Sources

In [ ]:
# Load core datasets directly — no dependency on src for maximum portability
orders      = pd.read_csv(os.path.join(DATA_DIR, 'orders.csv'), parse_dates=['order_date'])
order_items = pd.read_csv(os.path.join(DATA_DIR, 'order_items.csv'))
products    = pd.read_csv(os.path.join(DATA_DIR, 'products.csv'))
customers   = pd.read_csv(os.path.join(DATA_DIR, 'customers.csv'))
geography   = pd.read_csv(os.path.join(DATA_DIR, 'geography.csv'))
returns     = pd.read_csv(os.path.join(DATA_DIR, 'returns.csv'), parse_dates=['return_date'])
sales       = pd.read_csv(os.path.join(DATA_DIR, 'sales.csv'), parse_dates=['Date'])
web_traffic = pd.read_csv(os.path.join(DATA_DIR, 'web_traffic.csv'), parse_dates=['date'])
promotions  = pd.read_csv(os.path.join(DATA_DIR, 'promotions.csv'))

# Build transaction-level dataset
tx = order_items.merge(products, on='product_id', how='left')
tx = tx.merge(orders, on='order_id', how='left')
# Join geography via zip to get 'region'
geo_dedup = geography.drop_duplicates(subset='zip')
tx = tx.merge(geo_dedup[['zip', 'region']], on='zip', how='left')

# Compute revenue columns
tx['quantity'] = pd.to_numeric(tx['quantity'], errors='coerce').fillna(0)
tx['unit_price'] = pd.to_numeric(tx['unit_price'], errors='coerce').fillna(0)
tx['discount_amount'] = pd.to_numeric(tx['discount_amount'], errors='coerce').fillna(0)
tx['cogs'] = pd.to_numeric(tx['cogs'], errors='coerce').fillna(0)

tx['gross_revenue'] = tx['quantity'] * tx['unit_price']
tx['net_revenue'] = (tx['gross_revenue'] - tx['discount_amount']).clip(lower=0)
tx['line_cogs'] = tx['quantity'] * tx['cogs']

# Tag returned items
returned_orders = returns[['order_id', 'product_id']].drop_duplicates()
returned_orders['is_returned'] = 1
tx = tx.merge(returned_orders, on=['order_id', 'product_id'], how='left')
tx['is_returned'] = tx['is_returned'].fillna(0).astype(int)

print(f'Transactions loaded: {len(tx):,} rows, {tx.shape[1]} columns')
print(f'Columns: {tx.columns.tolist()}')

## 1 — Executive Summary: The North Star Metrics

In [ ]:
total_rev = tx['net_revenue'].sum()
total_cogs = tx['line_cogs'].sum()
total_profit = total_rev - total_cogs
margin = (total_profit / total_rev) * 100 if total_rev > 0 else 0
return_rate = tx['is_returned'].mean() * 100

print(f'Total Net Revenue : {total_rev:,.0f}')
print(f'Total Gross Profit: {total_profit:,.0f}')
print(f'Profit Margin     : {margin:.2f}%')
print(f'Return Rate       : {return_rate:.2f}%')

## 2 — Revenue Trend Over Time

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)
axes[0].plot(sales['Date'], sales['Revenue'], lw=0.6, color='steelblue')
axes[0].set_title('Daily Revenue (from sales.csv)', fontsize=14)
axes[0].set_ylabel('Revenue')
axes[1].plot(sales['Date'], sales['COGS'], lw=0.6, color='darkorange')
axes[1].set_title('Daily COGS', fontsize=14)
axes[1].set_ylabel('COGS')
plt.tight_layout()
plt.show()

## 3 — Geographic Profitability: Where is the Money?

In [ ]:
geo_revenue = tx.groupby('region')['net_revenue'].sum().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
sns.barplot(x=geo_revenue.index, y=geo_revenue.values, ax=ax, palette='viridis')
ax.set_title('Net Revenue by Region', fontsize=14)
ax.set_ylabel('Net Revenue')
ax.set_xlabel('Region')
for i, v in enumerate(geo_revenue.values):
    ax.text(i, v, f'{v/1e6:.1f}M', ha='center', va='bottom', fontsize=9)
plt.tight_layout()
plt.show()

print('\nInsight: Focus marketing budget on the top-revenue regions.')

## 4 — Product & Inventory: Return Rate by Category

In [ ]:
cat_returns = tx.groupby('category')['is_returned'].mean().sort_values(ascending=False)
fig, ax = plt.subplots(figsize=(10, 5))
cat_returns.plot(kind='barh', color='salmon', ax=ax)
ax.set_title('Return Rate by Product Category', fontsize=14)
ax.set_xlabel('Return Probability')
plt.tight_layout()
plt.show()

top_return_cat = cat_returns.idxmax()
print(f'\nHighest return rate category: {top_return_cat} ({cat_returns.max():.2%})')

## 5 — Deep Dive: Return Reasons

In [ ]:
# Load return reasons directly from returns.csv (not in tx aggregation)
reason_dist = returns['return_reason'].value_counts()

fig, ax = plt.subplots(figsize=(8, 8))
reason_dist.plot(kind='pie', autopct='%1.1f%%', ax=ax, startangle=90)
ax.set_ylabel('')
ax.set_title('Distribution of All Return Reasons', fontsize=14)
plt.tight_layout()
plt.show()

top_reason = reason_dist.idxmax()
top_pct = reason_dist.max() / reason_dist.sum() * 100
print(f'\nTop return reason: \"{top_reason}\" ({top_pct:.1f}%)')
print('Prescriptive Action: Address the #1 return reason to directly reduce refund costs.')

## 6 — Promotion Effectiveness

In [ ]:
# Compare revenue with vs without promotion
tx['has_promo'] = tx['promo_id'].notna() & (tx['promo_id'] != 'NO_PROMO')
promo_compare = tx.groupby('has_promo')['net_revenue'].agg(['mean', 'sum', 'count'])
promo_compare.index = ['No Promo', 'With Promo']
print('Revenue by Promotion Status:')
print(promo_compare.to_string())

fig, ax = plt.subplots(figsize=(8, 5))
promo_compare['mean'].plot(kind='bar', color=['#636EFA', '#EF553B'], ax=ax)
ax.set_title('Average Line Revenue: Promo vs No Promo', fontsize=14)
ax.set_ylabel('Avg Net Revenue per Line Item')
ax.set_xticklabels(ax.get_xticklabels(), rotation=0)
plt.tight_layout()
plt.show()

## 7 — Web Traffic Synergy: Sessions vs. Revenue

In [ ]:
# Aggregate web traffic daily and merge with sales
traffic_daily = web_traffic.groupby('date').agg(
    sessions=('sessions', 'sum'),
    unique_visitors=('unique_visitors', 'sum')
).reset_index()

merged = sales.merge(traffic_daily, left_on='Date', right_on='date', how='inner')

fig, ax = plt.subplots(figsize=(10, 6))
sns.regplot(data=merged, x='sessions', y='Revenue', scatter_kws={'alpha': 0.3, 's': 10}, ax=ax)
ax.set_title('Correlation: Web Sessions vs. Daily Revenue', fontsize=14)
plt.tight_layout()
plt.show()

corr = merged[['sessions', 'Revenue']].corr().iloc[0, 1]
print(f'Pearson Correlation: {corr:.3f}')

## 8 — Monthly Seasonality

In [ ]:
sales['month'] = sales['Date'].dt.month
monthly_avg = sales.groupby('month')[['Revenue', 'COGS']].mean()

fig, ax = plt.subplots(figsize=(10, 5))
monthly_avg.plot(kind='bar', ax=ax, color=['steelblue', 'darkorange'])
ax.set_title('Average Daily Revenue & COGS by Month', fontsize=14)
ax.set_xlabel('Month')
ax.set_ylabel('Average Value')
ax.set_xticklabels(range(1, 13), rotation=0)
plt.tight_layout()
plt.show()

peak_month = monthly_avg['Revenue'].idxmax()
print(f'\nPeak revenue month: {peak_month}')
print('Prescriptive: Pre-stock inventory and increase ad spend 1 month before peak.')

## 9 — Prescriptive Recommendations

Based on the analysis above, Team GenCore recommends the following:

1. **Reduce Returns**: The #1 return reason should be addressed first (e.g., if `wrong_size` → improve the Size Guide).
2. **Geographic Focus**: Allocate more marketing budget to top-revenue regions for higher ROI.
3. **Promotion Strategy**: Evaluate whether promotions increase net revenue enough to justify the discount cost.
4. **Seasonal Preparation**: Pre-stock inventory and ramp up marketing spend 1 month before the peak revenue month.
5. **Web Traffic Optimization**: Strengthen SEO/SEM campaigns on days with historically high conversion rates.